# Level 2 — Inspect columns properly

I am checking unique values, category counts, and suspicious labels. This step is useful before deciding how to clean.

In [1]:
from pathlib import Path
import pandas as pd

# Works if I open Jupyter from the repo root or from notebooks/
data_path = Path("data/sales_raw.csv")
if not data_path.exists():
    data_path = Path("../data/sales_raw.csv")

df = pd.read_csv(data_path)
df.head(3)

,line_id,order_id,order_date,customer_id,city,category,product,quantity,unit_price_pkr,discount_pct,revenue_pkr,payment_method,order_status,channel,delivery_days
0,LN-00001,ORD-2025-25001,2025-05-13,C-1023,Lahore,Office,Whiteboard,4,6000,10%,21600.0,Card,Delivered,Store,10.0
1,LN-00002,ORD-2025-25001,2025-08-19,C-1134,Islamabad,Home,Wall Clock,5,3100,15%,13175.0,Cash,Delivered,Store,8.0
2,LN-00003,ORD-2025-25002,2025-09-10,C-1052,Rawalpindi,NaN,Desk Lamp,3,2592,0%,7776.0,Bank Transfer,Delivered,Online,6.0


### Unique values versus number of unique values

In [2]:
print("Channels:", df["channel"].unique())
print("No. of channels:", df["channel"].nunique())
print("No. of customers:", df["customer_id"].nunique())

Channels: ['Store' 'Online' 'WhatsApp']
No. of channels: 3
No. of customers: 172


### Counts and percentages

In [3]:
df["order_status"].value_counts(dropna=False)

order_status
Delivered    469
Pending       55
Cancelled     29
Returned      12
Name: count, dtype: int64

In [4]:
(df["channel"].value_counts(normalize=True, dropna=False) * 100).round(1)

channel
Online      48.5
Store       36.1
WhatsApp    15.4
Name: proportion, dtype: float64

### Dirty categories
Here the same city might be typed in multiple ways. I want to *spot* the problem first, not blindly replace everything.

In [5]:
df["city"].value_counts(dropna=False).head(20)

city
Lahore        140
Karachi       105
Islamabad      92
Rawalpindi     81
Faisalabad     56
Lhr            16
KHI            15
LAHORE         14
lahore         14
NaN            13
 Lahore         7
karachi         7
 Karachi        5
Name: count, dtype: int64

In [6]:
df["category"].value_counts(dropna=False)

category
Electronics    201
Office         131
Home           127
Accessories     88
NaN             18
Name: count, dtype: int64

### Cardinality check

In [7]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique": df.nunique(dropna=True),
    "missing": df.isna().sum()
})
summary.sort_values("unique", ascending=False)

,dtype,unique,missing
line_id,object,560,0
revenue_pkr,float64,321,15
order_date,object,304,0
order_id,object,280,0
customer_id,object,172,9
unit_price_pkr,int64,50,0
product,object,17,0
quantity,object,12,0
city,object,12,13
delivery_days,float64,11,117


### Rare values or possible typos

In [8]:
city_counts = df["city"].value_counts(dropna=False)
city_counts[city_counts <= 8]

city
 Lahore      7
karachi      7
 Karachi     5
Name: count, dtype: int64

**My note:** `line_id` is expected to have very high cardinality. `channel` is expected to be low. High cardinality by itself is not a data quality problem.